#### When to use json.loads() with widgets?

|    parameter   |           Example                    |
|----------------|--------------------------------------|
| excluded_codes | ["BR~", "MK~", "CID~", "CN~", "FF~"] |
|   json_loads   | ["A1", "A2",]                        |
|  literal_eval  | false                                |
| overwrite_enable | false                              |
| root_folder_name | bronze_internal                    |
| source_col       | [ "cust_id", "cust_sub_manager_name", "cust_manager", "cust_manager_id", "cust_sub_manager_id", "cust_desc_name", "advertise_channel_type", "sales_id"] |
| source_database_names | {"bronze": "sales_profit_ration", "silver": "sales_profit_ration", "manual": "dim_manual_data", "gold": "fact_sales_data"} |
| source_table_names    | {"country": "db_country_base", "source": "sales_part_data",  "sales": "dim_sales_dep_metadata", "detail": "dim_sales_g220_dep_detail", "fact": "fact_sales_g220_dep_detail", "key_value": "sales_data_dictionary", "currency_path": "currency"} |
| write_options_dict    | {"mergeSchema":"true"} |

**Content:**
- `Why json.loads is needed?`
- `Widget contains a LIST of database names`
- `Widget contains a DICTIONARY`
- `Widget value with escaped quotes`
- `Realtime Scenario`

     source_database_names = json.loads(dbutils.widgets.get("source_database_names"))

|    Syntax     |  Description    |
|---------------|-----------------|
| **dbutils.widgets.get("source_database_names")** | Reads a Databricks widget value (**always returns a string**) |
| **json.loads(...)**  | Converts that **string** into a Python **list or dictionary** |

##### 1) Using json.loads()
- Works only when the widget value is **valid JSON**.

- **Success:**
  - `clean JSON`
  - `double quotes`
  - `no trailing commas`
  
- **Fails** immediately if:
  - **single quotes ' '**
  - Python-style **booleans** `True` & `False`.
  - **trailing commas**.

| JSON Requirement           |      Input           |     Output           |
| -------------------------- | -------------------- | -------------------- |
| **Double quotes** only     | `["A101", "A102"]`   |  ['A101', 'A102']    |
| No **single quotes**       | `['A101','A102']`    |        ❌           |
| **True/False → lowercase** | `true`, `false`      |  `True`, `False`     |
|                            | `True`, `False`      |  ❌, ❌             |
| **Null → lowercase**       | `null`               |                      |
|                            | `Null`               |  ❌                 |
| **trailing commas**        | ["A1", "A2",]        |  ❌                 |

**What is a trailing comma?**

     ["A1", "A2",]
         (or)
     {
       "id": 1,
       "name": "Sam",
     }

**ast.literal_eval handles trailing commas**

     import ast
     ast.literal_eval("['A1', 'A2',]")

| Feature                          | `json.loads()` | `ast.literal_eval()`       |
| -------------------------------- | -------------- | -------------------------- |
| Accepts single quotes            | ❌ No           | ✔ Yes                      |
| Accepts trailing commas          | ❌ No           | ✔ Yes                      |
| Accepts Python booleans (`True`) | ❌ No           | ✔ Yes                      |
| Accepts JSON only                | ✔ Yes          | ❌ No (Python objects only) |
| Safe to evaluate                 | ✔ Yes          | ✔ Yes                      |


**Why json.loads is needed?**
- `Widgets return always string`.

     # So even if widget looks like a list
           ["db1", "db2"]

     # Without json.loads, Python treats it as:
           "[\"db1\", \"db2\"]"  # string, not list
     
     # json.loads() converts it into a real list.

##### 1) Widget contains a LIST of database names

In [0]:
%skip
dbutils.widgets.removeAll()

In [0]:
import json

In [0]:
dbutils.widgets.text("json_loads", "", "json_loads")

json_loads = json.loads(dbutils.widgets.get("json_loads"))
print(json_loads)
print(type(json_loads))

{'bronze': 'sales_profit_ration', 'silver': 'sales_profit_ration', 'manual': 'dim_manual_data', 'gold': 'fact_sales_data'}
<class 'dict'>


In [0]:
# Create the widget
dbutils.widgets.text("source_database_names_list", '["db1", "db2", "db3"]', "source_database_names_list")
names_wo_json = dbutils.widgets.get("source_database_names_list")
print("names_wo_json: ", names_wo_json)
print(type(names_wo_json))

names_w_json = json.loads(dbutils.widgets.get("source_database_names_list"))
print("names_w_json: ", names_w_json)
print(type(names_w_json))

names_wo_json:  {"bronze": "sales_profit_ration", "silver": "sales_profit_ration", "manual": "dim_manual_data", "gold": "fact_sales_data"}
<class 'str'>
names_w_json:  {'bronze': 'sales_profit_ration', 'silver': 'sales_profit_ration', 'manual': 'dim_manual_data', 'gold': 'fact_sales_data'}
<class 'dict'>


##### 2) Widget contains a DICTIONARY

In [0]:
# Read the widget
import json

# Returns the widget value as a string
names_wo_json = dbutils.widgets.get("source_database_names")
print("names_wo_json: ", names_wo_json)
print(type(names_wo_json))

# Parses the string into a Python object (list or dict)
names_w_json = json.loads(dbutils.widgets.get("source_database_names"))
print("\nnames_w_json: ", names_w_json)
print(type(names_w_json))

names_wo_json:  {"bronze": "db_f220", "silver": "db_g220", "gold": "sales_data"}
<class 'str'>

names_w_json:  {'bronze': 'db_f220', 'silver': 'db_g220', 'gold': 'sales_data'}
<class 'dict'>


In [0]:
for db in names_wo_json:
    print("Processing:", db)

Processing: {
Processing: "
Processing: b
Processing: r
Processing: o
Processing: n
Processing: z
Processing: e
Processing: "
Processing: :
Processing:  
Processing: "
Processing: d
Processing: b
Processing: _
Processing: f
Processing: 2
Processing: 2
Processing: 0
Processing: "
Processing: ,
Processing:  
Processing: "
Processing: s
Processing: i
Processing: l
Processing: v
Processing: e
Processing: r
Processing: "
Processing: :
Processing:  
Processing: "
Processing: d
Processing: b
Processing: _
Processing: g
Processing: 2
Processing: 2
Processing: 0
Processing: "
Processing: ,
Processing:  
Processing: "
Processing: m
Processing: a
Processing: n
Processing: u
Processing: a
Processing: l
Processing: "
Processing: :
Processing:  
Processing: "
Processing: d
Processing: b
Processing: _
Processing: m
Processing: a
Processing: n
Processing: u
Processing: a
Processing: l
Processing: _
Processing: d
Processing: a
Processing: t
Processing: a
Processing: "
Processing: ,
Processing:  
Proces

In [0]:
for key in names_w_json:
    print(key)

bronze
silver
manual
gold


In [0]:
for key, value in names_w_json.items():
    print(f"{key}:{value}")

bronze:db_f220
silver:db_g220
manual:db_manual_data
gold:sales_data


In [0]:
dbutils.widgets.text("source_database_names_dict", '{"prod": "sales_db", "dev": "sales_db_dev"}')

In [0]:
import json

configs = json.loads(dbutils.widgets.get("source_database_names_dict"))
print(configs)
print(type(configs))

print(configs["prod"])
print(configs["dev"])

{'prod': 'sales_db', 'dev': 'sales_db_dev'}
<class 'dict'>
sales_db
sales_db_dev


##### 3) Widget value with escaped quotes

In [0]:
# Create the widget
dbutils.widgets.text("source_database_names_text", '[\"db1\", \"db2\"]')

In [0]:
names_text_wo_json_loads = dbutils.widgets.get("source_database_names_text")
print(names_text_wo_json_loads)
print(type(names_text_wo_json_loads))

names_text_w_json_loads = json.loads(dbutils.widgets.get("source_database_names_text"))
print("\n", names_text_w_json_loads)
print(type(names_text_w_json_loads))

["db1", "db2"]
<class 'str'>

 ['db1', 'db2']
<class 'list'>


##### 4) Realtime Scenario

In [0]:
%skip
# list of the source table names
source_table_names = {"country": "db_country_base", "source": "sales_part_data",  "target": "dim_sales_dep_metadata", "detail": "dim_sales_g220_dep_detail", "fact": "fact_sales_g220_dep_detail", "key_value": "sales_data_dictionary", "currency_path": "currency"}

# list of the source database names
source_database_names = {"bronze": "db_f220", "silver": "db_g220", "gold": "sales_data"}

rename_columns  = [["start_date", "START_DT"], ["end_date", "END_DT"], ["reach", "REACH"], ["report_date", "REPORT_DT"], ["clicks", "CLICKS"], ["revenue", "REVENUE_AMT"], ["load_date", "LOAD_DATE"] ]

# ADLS location
adls_folder_name = bm_g220_sales

# list of columns that are required for sales table
source_col = [ "cust_id", "cust_sub_manager_name", "cust_manager", "cust_manager_id", "cust_sub_manager_id", "cust_desc_name", "advertise_channel_type", "sales_id"]

# dictionaries & list
read_options_dict = {"header": "true"}
write_options_dict = {"mergeSchema":"true"}

overwrite_enable = false

In [0]:
source_table_names = {"country": "db_country_base", "source": "sales_part_data",  "target": "dim_sales_dep_metadata", "detail": "dim_sales_g220_dep_detail", "fact": "fact_sales_g220_dep_detail", "key_value": "sales_data_dictionary", "currency_path": "currency"}

source_database_names = {"bronze": "db_f220", "silver": "db_g220", "gold": "sales_data"}

rename_columns = [["start_date", "START_DT"], ["end_date", "END_DT"], ["reach", "REACH"], ["report_date", "REPORT_DT"], ["clicks", "CLICKS"], ["revenue", "REVENUE_AMT"], ["load_date", "LOAD_DATE"] ]

adls_folder_name = "bm_g220_sales"

source_col = [ "cust_id", "cust_sub_manager_name", "cust_manager", "cust_manager_id", "cust_sub_manager_id", "cust_desc_name", "advertise_channel_type", "sales_id"]

read_options_dict = {"header": "true"}
write_options_dict = {"mergeSchema":"true"}
overwrite_enable = "false"

In [0]:
import json

# Declare Inputs Widgets
dbutils.widgets.text("source_table_names", "", "source_table_names")
dbutils.widgets.text("source_database_names", "", "source_database_names")
dbutils.widgets.text("rename_columns", "", "rename_columns")
dbutils.widgets.text("adls_folder_name", "", "adls_folder_name")
dbutils.widgets.text("source_col", "", "source_col")
dbutils.widgets.text("read_options_dict", "", "read_options_dict")
dbutils.widgets.text("schedule_date", "", "schedule_date")
dbutils.widgets.text("write_options_dict", "", "write_options_dict")
dbutils.widgets.text("overwrite_enable", "", "overwrite_enable")

# Extract Inputs From Widgets
source_table_names    = json.loads(dbutils.widgets.get("source_table_names"))
source_database_names = json.loads(dbutils.widgets.get("source_database_names"))
rename_columns        = json.loads(dbutils.widgets.get("rename_columns"))
adls_folder_name      = dbutils.widgets.get("adls_folder_name")
source_col            = json.loads(dbutils.widgets.get("source_col"))
read_options_dict     = json.loads(dbutils.widgets.get("read_options_dict"))
schedule_date         = dbutils.widgets.get("schedule_date")
write_options_dict    = json.loads(dbutils.widgets.get("write_options_dict"))
overwrite_enable      = json.loads(dbutils.widgets.get("overwrite_enable"))


print("source_table_names: ", source_table_names)
print(type(source_table_names))

print("\nsource_database_names: ", source_database_names)
print(type(source_database_names))

print("\nrename_columns: ", rename_columns)
print(type(rename_columns))

print("\nadls_folder_name: ", adls_folder_name)
print(type(adls_folder_name))

print("\nsource_col: ", source_col)
print(type(source_col))

print("\nread_options_dict: ", read_options_dict)
print(type(read_options_dict))

print("\nschedule_date: ", schedule_date)
print(type(schedule_date))

print("\nwrite_options_dict: ", write_options_dict)
print(type(write_options_dict))

print("\noverwrite_enable: ", overwrite_enable)
print(type(overwrite_enable))

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File <command-5165860683365732>, line 15
     12 dbutils.widgets.text("overwrite_enable", "", "overwrite_enable")
     14 # Extract Inputs From Widgets
---> 15 source_table_names    = json.loads(dbutils.widgets.get("source_table_names"))
     16 source_database_names = json.loads(dbutils.widgets.get("source_database_names"))
     17 rename_columns        = json.loads(dbutils.widgets.get("rename_columns"))

File /usr/lib/python3.12/json/__init__.py:346, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    341     s = s.decode(detect_encoding(s), 'surrogatepass')
    343 if (cls is None and object_hook is None and
    344         parse_int is None and parse_float is None and
    345         parse_constant is None and object_pairs_hook is None and not kw):
--> 346     return _default_decoder.de

In [0]:
# data sources
country_table_name   = source_table_names["country"]
source_table_name    = source_table_names["source"]
target_table_name    = source_table_names["target"]
key_value_table_name = source_table_names["key_value"]
bronze_database_name = source_database_names["bronze"]
silver_database_name = source_database_names["silver"]
gold_database_name    = source_database_names["gold"]

print("\ncountry_table_name: ", country_table_name)
print("\nsource_table_name: ", source_table_name)
print("\ntarget_table_name: ", target_table_name)
print("\nkey_value_table_name: ", key_value_table_name)
print("\nbronze_database_name: ", bronze_database_name)
print("\nsilver_database_name: ", silver_database_name)
print("\ngold_database_name: ", gold_database_name)


country_table_name:  db_country_base

source_table_name:  sales_part_data

target_table_name:  dim_sales_dep_metadata

key_value_table_name:  sales_data_dictionary

bronze_database_name:  db_f220

silver_database_name:  db_g220

gold_database_name:  sales_data
